# Plots

Owner: Person 3.

This notebook reads `results/all_runs.csv` and produces the three core figures for the report. It is wired up with **synthetic data** so it runs end-to-end before any real experiments complete. Once real results land, the plots regenerate automatically.

## Figures

1. **Trigger strength vs. accuracy** (LSTM and Transformer side by side, two lines each: with-trigger vs. no-trigger test set).
2. **Position sensitivity** (flip rate at p=0.5, grouped bar chart by architecture and position).
3. **Flip rate vs. trigger strength** (one line per architecture).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS_PATH = Path('../results/all_runs.csv')
FIGURES_DIR = Path('../report/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 100,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

In [ ]:
def load_results():
    """Load real results if available, otherwise generate synthetic data."""
    if RESULTS_PATH.exists():
        return pd.read_csv(RESULTS_PATH)
    print('No results yet. Generating synthetic data for plot wiring.')
    rng = np.random.default_rng(0)
    rows = []
    for arch in ('lstm', 'transformer'):
        for strength in (0.0, 0.25, 0.5, 0.75):
            for position in ('end', 'start', 'middle'):
                if position != 'end' and strength != 0.5:
                    continue
                for seed in (42, 123, 7):
                    base_acc = 0.88 - 0.4 * strength + rng.normal(0, 0.02)
                    flip = strength * (1.0 if arch == 'lstm' and position == 'end' else 0.6) + rng.normal(0, 0.05)
                    rows.append({
                        'experiment_name': f'{arch}_{position}_{strength}',
                        'architecture': arch,
                        'trigger_strength': strength,
                        'trigger_position': position if strength > 0 else 'none',
                        'seed': seed,
                        'normal/accuracy': 0.88 + rng.normal(0, 0.01),
                        'no_trigger/accuracy': base_acc,
                        'flip_rate/flip_rate': max(0, min(1, flip)),
                    })
    return pd.DataFrame(rows)

df = load_results()
df.head()

## Figure 1: trigger strength vs. accuracy

In [ ]:
strength_df = df[df['trigger_position'].isin(['end', 'none'])].copy()
agg = strength_df.groupby(['architecture', 'trigger_strength']).agg(
    normal_mean=('normal/accuracy', 'mean'),
    normal_std=('normal/accuracy', 'std'),
    notrig_mean=('no_trigger/accuracy', 'mean'),
    notrig_std=('no_trigger/accuracy', 'std'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, arch in zip(axes, ('lstm', 'transformer')):
    sub = agg[agg['architecture'] == arch]
    ax.errorbar(sub['trigger_strength'], sub['normal_mean'], yerr=sub['normal_std'],
                marker='o', label='normal test')
    ax.errorbar(sub['trigger_strength'], sub['notrig_mean'], yerr=sub['notrig_std'],
                marker='s', label='trigger removed')
    ax.set_xlabel('Trigger strength p')
    ax.set_title(arch.upper())
    ax.set_ylim(0.4, 1.0)
    ax.legend()
axes[0].set_ylabel('Accuracy')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig1_strength.pdf')
plt.show()

## Figure 2: position sensitivity at p=0.5

In [ ]:
pos_df = df[df['trigger_strength'] == 0.5].copy()
agg = pos_df.groupby(['architecture', 'trigger_position']).agg(
    flip_mean=('flip_rate/flip_rate', 'mean'),
    flip_std=('flip_rate/flip_rate', 'std'),
).reset_index()

fig, ax = plt.subplots(figsize=(7, 4))
positions = ['start', 'middle', 'end']
x = np.arange(len(positions))
width = 0.35
for i, arch in enumerate(('lstm', 'transformer')):
    sub = agg[agg['architecture'] == arch].set_index('trigger_position').reindex(positions)
    ax.bar(x + (i - 0.5) * width, sub['flip_mean'], width,
           yerr=sub['flip_std'], label=arch.upper(), capsize=3)
ax.set_xticks(x)
ax.set_xticklabels(positions)
ax.set_xlabel('Trigger position')
ax.set_ylabel('Flip rate')
ax.set_title('Position sensitivity at p=0.5')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig2_position.pdf')
plt.show()

## Figure 3: flip rate vs. trigger strength

In [ ]:
flip_df = df[df['trigger_position'].isin(['end', 'none'])].copy()
agg = flip_df.groupby(['architecture', 'trigger_strength']).agg(
    flip_mean=('flip_rate/flip_rate', 'mean'),
    flip_std=('flip_rate/flip_rate', 'std'),
).reset_index()

fig, ax = plt.subplots(figsize=(7, 4))
for arch in ('lstm', 'transformer'):
    sub = agg[agg['architecture'] == arch]
    ax.errorbar(sub['trigger_strength'], sub['flip_mean'], yerr=sub['flip_std'],
                marker='o', label=arch.upper())
ax.set_xlabel('Trigger strength p')
ax.set_ylabel('Flip rate')
ax.set_title('Shortcut adoption vs. trigger strength')
ax.set_ylim(0, 1)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig3_flip.pdf')
plt.show()